In [ ]:
# Instala a integracao do LangChain com Groq para usar o modelo via API.
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


#Agents

In [ ]:
# Informe sua chave da API Groq para autenticar as chamadas.
api_key = "sua api key"

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
# Configura o modelo base que o agente usara para raciocinar e responder.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2, # 0.7 deixaria as respostas mais diversas
    api_key=api_key

)

In [ ]:
llm.invoke("hello")

AIMessage(content="Hello. It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 36, 'total_tokens': 61, 'completion_time': 0.04379038, 'completion_tokens_details': None, 'prompt_time': 0.000930682, 'prompt_tokens_details': None, 'queue_time': 0.173989583, 'total_time': 0.044721062}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0865-932b-72a1-aa93-b295ff09bc64-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 25, 'total_tokens': 61})

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


# @tool transforma a funcao em uma ferramenta que o agente pode chamar.
@tool
def pegar_clima(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Ta Calor 40C"


# Cria um agente com acesso a ferramenta de clima.
agent = create_agent(llm, tools=[pegar_clima])

In [ ]:
# Envia o contexto da conversa para o agente (mensagens de system + pergunta do usuario).
result = agent.invoke(
    {"messages": [

        {"role": "system", "content": "voce e um agent de IA especializado em ajudar o usuario"},
        {"role": "system", "content": "seu nome e Joao"},
        {"role": "user", "content": "qual seu nome?"}

    ]}
)

In [ ]:
result

{'messages': [SystemMessage(content='voce e um agent de IA especializado em ajudar o usuario', additional_kwargs={}, response_metadata={}, id='9791d767-0ea1-4453-9757-e013ffd1376b'),
  SystemMessage(content='seu nome e Joao', additional_kwargs={}, response_metadata={}, id='310c6fe1-ff61-4bfa-8777-c0e0aa2d97c3'),
  HumanMessage(content='qual seu nome?', additional_kwargs={}, response_metadata={}, id='8e013bc2-cd3c-40d3-857c-1f3195634b94'),
  AIMessage(content='Meu nome é Joao. Estou aqui para ajudar com qualquer coisa que você precise. Como posso ajudar você hoje?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 241, 'total_tokens': 270, 'completion_time': 0.060818807, 'completion_tokens_details': None, 'prompt_time': 0.023408995, 'prompt_tokens_details': None, 'queue_time': 0.173618212, 'total_time': 0.084227802}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reaso

In [ ]:
{'messages': [SystemMessage(content='voce e um agent de IA especializado em ajudar o usuario', additional_kwargs={}, response_metadata={}, id='6fc427de-be21-4052-8eb6-d086ce8d339e'),

              HumanMessage(content='qual a temperatura no RJ?', additional_kwargs={}, response_metadata={}, id='d57212d6-0341-487d-81e5-898a0487673a'),

              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8x471tp76', 'function': {'arguments': '{"location":"Rio de Janeiro"}', 'name': 'pegar_clima'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 236, 'total_tokens': 255, 'completion_time': 0.048513785, 'completion_tokens_details': None, 'prompt_time': 0.025370752, 'prompt_tokens_details': None, 'queue_time': 0.008321208, 'total_time': 0.073884537}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d086e-828b-7421-830f-09e8992ca917-0', tool_calls=[{'name': 'pegar_clima', 'args': {'location': 'Rio de Janeiro'}, 'id': '8x471tp76', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 19, 'total_tokens': 255}),

              ToolMessage(content='Weather in Rio de Janeiro: Ta Calor 40C', name='pegar_clima', id='36acf9e2-a70f-40b3-ab48-593ced4ae211', tool_call_id='8x471tp76'),

              AIMessage(content='A temperatura no Rio de Janeiro é de 40°C.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 277, 'total_tokens': 290, 'completion_time': 0.03229442, 'completion_tokens_details': None, 'prompt_time': 0.02686717, 'prompt_tokens_details': None, 'queue_time': 0.139582122, 'total_time': 0.05916159}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d086e-834b-70d2-9f5c-d951f57db529-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 13, 'total_tokens': 290})]}

In [ ]:
for x in agent.stream(
    {"messages": [

        {"role": "system", "content": "voce e um agent de IA especializado em ajudar o usuario"},
        {"role": "user", "content": "qual tempo em sao paulo?"}

        ]}, stream_mode = "messages"):
  print(x)

(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'groq'}, id='lc_run--019d07f0-0946-75b0-8887-03707aa66f73', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'model', 'langgraph_triggers': ('branch:to:model',), 'langgraph_path': ('__pregel_pull', 'model'), 'langgraph_checkpoint_ns': 'model:a198176d-b91e-7231-fd65-10ecc6f15c31', 'checkpoint_ns': 'model:a198176d-b91e-7231-fd65-10ecc6f15c31', 'ls_provider': 'groq', 'ls_model_name': 'llama-3.3-70b-versatile', 'ls_model_type': 'chat', 'ls_temperature': 0.2})
(AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'r2g4kawa5', 'function': {'arguments': '{"location":"Sao Paulo"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'model_provider': 'groq'}, id='lc_run--019d07f0-0946-75b0-8887-03707aa66f73', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Sao Paulo'}, 'id': 'r2g4kawa5', 'type': 'tool_call'}

In [ ]:
#!pip install tavily langchain-google-genai

In [ ]:
from tavily import TavilyClient
from typing import Literal

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview", api_key="SUA CHAVE DE API DO GOOGLE")

In [ ]:
llm.invoke("hello")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EqIGCp8GAb4+9vupTXpbS8fctoOQTjyM2DazBRFQ2Vfu9DWlHg99lrsR6qp2whtth0UvkjQ9xpOAH868sFGPdxARPvlvbNH+0Bsl7mKz9Kuk/Mk7WStkrCdbhlOBscGDA7uBb15zHh9Atk2o/3N/NR2DgvOr9bWctYEbGMnw5Wl/5I6nhk4rxXTpxwHi25uuP/K4eQbMQahG0xsAqQSNWsM5FP7bklcvBMJoBiQcFsp61+oNpVVp3FcN/Kp0CCWIXYrznBNndNoNtQfy8otUKN4odmy/wHSsNWLNB8QFev2NkSq0KdHSfgbd+7YKTQIJ+NwwmLjJD5ppWIJIC1NYUMs/rLuHFv7ggV5Kg5RAl2eIJllUa2qVIjGWKim0iTz0ZzxpuJN1rf4GwaboX+B7p67RZFTKKyRmDjkXtTlinegNssCf+J66fpRHZBKZNMvW+CAS8ydZv7KoWKC5jKopzwT50LysYPAg1xsEen7b6y1T7w5XG1f5mnBNiI1si6WFrOaxv+kbyBvIjSuqpiSEN9J9+UF5eb7lzIELgsok9RmyGqKIqPzkNqJew+eErjwpmk2Nl2CqAVz8dBRuB777c3Ae4Xdxc1wcjdoxs2knMepbfP14U6Cd8hpZDumO6uNwH9RplPjPRQ4+ZIraLpr64TBDBdKKeQLfEh5cl1HqZX/sEItNau6OtL19+vNRqNXqaFmfH2c0cOZVlVsv5/nG5wpcoxMITRf0DFzv8zzBUQNV03aPE02mzekiTF3mOAlUEBnwYWs2to9CHseMrwcf56VE89y04mW0bg5PxsRc/sB/R/4Y0RjBAf9ENe9W4S6V9PTXDovg+Hgon+ilh1UcC6zCtTdAWAT2xiXoQpjolScbA8Ciy7nNK4WrzTv89pc

In [ ]:
tavily_client = TavilyClient(api_key="SUA CHAVE DE API DO TAVILY")

@tool
def internet_search(
    query: str) -> str:

    """Run a web search"""
    return str(tavily_client.search(
        query,
        max_results=5,
        include_raw_content=True,

    ))



@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"


agent = create_agent(llm, tools=[get_weather, internet_search])

In [ ]:
result = agent.invoke(
    {"messages": [

        {"role": "system", "content": "voce e um agent de IA especializado em ajudar o usuario"},
        {"role": "system", "content": "seu nome e Joao"},
         {"role": "user", "content": "qual o preco do bitcoin?"}

        ]})

In [ ]:
result

{'messages': [SystemMessage(content='voce e um agent de IA especializado em ajudar o usuario', additional_kwargs={}, response_metadata={}, id='06d4453e-bcd5-49cc-9c4f-c3f9d5055f0c'),
  SystemMessage(content='seu nome e Joao', additional_kwargs={}, response_metadata={}, id='202560a4-a189-4302-bbd6-542a272b1d62'),
  HumanMessage(content='qual o preco do bitcoin?', additional_kwargs={}, response_metadata={}, id='5e6c3065-d687-4353-a4f3-5c78234bd792'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'internet_search', 'arguments': '{"query": "pre\\u00e7o do bitcoin hoje"}'}, '__gemini_function_call_thought_signatures__': {'6486536b-681a-4a73-9694-7301199c77e6': 'EpkFCpYFAb4+9vuZhBiz6zbZLfzxydEmJ56t4nXri57o5HI8B9tepaEQ3vRcwebcB/ai/fLGUHtdluYeuk689N9+2GZfy+3BceQc5DXUx2RHkkcDMy+2HZ1ez8hIS+tEmRQ/xEJ6pF157fiQsAML8LaJeHfuhjVhrXlw+NfCH4+MXKRtHQMqu1m0RDjQy7VFOfLMPYbrH2VO39Zd5LJ9afeFpUZkYt9qaOgzCd+qnBoH6Wp8k8yBYiivvKfGhMJ8nvTjgWX/yI+IEuja7BSTzRE4vGRiq6M5k0Hqa1JAHslpumFGbAJU/Taj